# 02a · A Neural Network from Scratch (NumPy) `[local]`

**Goal:** build, train and evaluate a neural network that reads handwritten digits, using **only NumPy**. After this, nothing in PyTorch is magic: it's the same math, automated.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | Neurons as weighted votes, learning as walking downhill, backprop as blame assignment |
| 📐 Math | Forward pass, softmax, cross-entropy, backprop for every layer, with shapes |
| 🔨 From scratch | A 2-layer network: forward, backward, gradient check, training loop |
| 🚀 Real tool | Verify your gradients against PyTorch's autograd |
| 📊 Evaluate | Train/val/test discipline, confusion matrix, per-class accuracy, error gallery |
| 🧪 Experiment | Learning rate, no ReLU, zero init, hidden size, overfitting |

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import time
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets
from cvlib.viz import plot_history

rng = np.random.default_rng(42)

---
# 🧠 1 · Intuition

### A neuron is a weighted vote
A neuron looks at its inputs (here, 784 pixel brightnesses), multiplies each by a **weight** ("how much do I care about this pixel, and in which direction?"), adds them up plus a **bias**, and passes the result through an **activation function**.

A neuron for "is this a 1?" might learn big positive weights for pixels down the middle and negative weights for pixels on the sides.

### A network is layers of votes
- **Hidden layer:** 128 neurons each learn a different useful pattern (a stroke, a loop, a corner).
- **Output layer:** 10 neurons, one per digit, combine those patterns into a score for each digit.
- **ReLU** (`max(0, x)`) between layers is what makes this more powerful than one big weighted sum. Without it, two layers collapse into one (Experiment 2 proves it).

### Learning = walking downhill in fog
The **loss** measures how wrong the network is. Picture it as a landscape where every position is one setting of all the weights. You're standing in fog and can only feel the slope under your feet: that's the **gradient**. Take a small step downhill. Repeat thousands of times. That's **gradient descent**. The step size is the **learning rate**.

### Backpropagation = assigning blame
After a wrong answer, each weight needs to know: *"how much did I contribute to this mistake?"* Backprop answers this by passing the error **backwards** through the network, layer by layer, using the chain rule. Each layer only needs the blame from the layer after it.

---
# 📐 2 · The math (with shapes)

Batch of $B$ images, flattened to 784 numbers each. Hidden size $H = 128$, $C = 10$ classes.

**Forward pass**

| Step | Formula | Shape |
|---|---|---|
| input | $X$ | $(B, 784)$ |
| hidden pre-activation | $Z_1 = X W_1 + b_1$ | $(B, H)$ |
| hidden activation | $A_1 = \text{ReLU}(Z_1) = \max(0, Z_1)$ | $(B, H)$ |
| output scores ("logits") | $Z_2 = A_1 W_2 + b_2$ | $(B, C)$ |
| probabilities | $P = \text{softmax}(Z_2)$, i.e. $P_{ic} = \dfrac{e^{Z_{ic}}}{\sum_k e^{Z_{ik}}}$ | $(B, C)$ |
| loss | $L = -\dfrac{1}{B}\sum_i \log P_{i, y_i}$ (cross-entropy) | scalar |

**Cross-entropy, intuitively:** the penalty is $-\log(\text{probability given to the correct answer})$. Confidently right ($p = 0.99$) costs 0.01; confidently wrong ($p = 0.01$) costs 4.6.

**Backward pass:** the chain rule, one layer at a time. With $Y$ = one-hot labels:

| Gradient | Formula | Why |
|---|---|---|
| $\partial L / \partial Z_2$ | $(P - Y) / B$ | softmax + cross-entropy simplify beautifully |
| $\partial L / \partial W_2$ | $A_1^\top \, \partial Z_2$ | each weight's input × the blame at its output |
| $\partial L / \partial b_2$ | $\sum_{\text{batch}} \partial Z_2$ | bias input is always 1 |
| $\partial L / \partial A_1$ | $\partial Z_2 \, W_2^\top$ | blame flows back through the weights |
| $\partial L / \partial Z_1$ | $\partial A_1 \odot \mathbb{1}[Z_1 > 0]$ | ReLU passes blame only where it was active |
| $\partial L / \partial W_1$ | $X^\top \, \partial Z_1$ | same pattern as layer 2 |

**Update** (stochastic gradient descent): $W \leftarrow W - \eta \, \partial L / \partial W$ with learning rate $\eta$.

**Initialization:** weights start small and random. **He initialization** ($\text{std} = \sqrt{2 / \text{fan\_in}}$) keeps signals from shrinking or exploding as they pass through ReLU layers.

---
# 3 · Data: MNIST, and the train / validation / test rule

60,000 training and 10,000 test images of handwritten digits, 28×28 grayscale.

**The rule you'll follow for the rest of the course:**
- **train** (50,000): the network learns from these
- **validation** (10,000, carved out of train): you use these to make choices (learning rate, size, when to stop)
- **test** (10,000): touched **once**, at the very end, to report the final number

If you tune anything while looking at test accuracy, the test set becomes part of training, and your reported number is optimistic.

In [ ]:
train_full = datasets.MNIST(env.data_dir, train=True, download=True)
test_set = datasets.MNIST(env.data_dir, train=False, download=True)

X_all = train_full.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0   # scale pixels to 0..1
y_all = train_full.targets.numpy()
X_test = test_set.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
y_test = test_set.targets.numpy()

perm = rng.permutation(len(X_all))                     # shuffle before splitting!
train_idx, val_idx = perm[:50_000], perm[50_000:]
X_train, y_train = X_all[train_idx], y_all[train_idx]
X_val, y_val = X_all[val_idx], y_all[val_idx]
print(f"train {X_train.shape} | val {X_val.shape} | test {X_test.shape}")
print("class counts in train:", np.bincount(y_train))

fig, axes = plt.subplots(2, 10, figsize=(12, 2.8))
for ax, i in zip(axes.flat, rng.choice(len(X_train), 20, replace=False)):
    ax.imshow(X_train[i].reshape(28, 28), cmap="gray"); ax.set_title(y_train[i]); ax.axis("off")
plt.suptitle("Always look at your data first"); plt.tight_layout(); plt.show()

---
# 🔨 4 · Build it from scratch

In [ ]:
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)              # subtract the max: same result, no overflow in exp()
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy(P, y):
    return -np.mean(np.log(P[np.arange(len(y)), y] + 1e-12))

class TwoLayerNet:
    def __init__(self, n_in=784, n_hidden=128, n_out=10, init="he", seed=0):
        r = np.random.default_rng(seed)
        if init == "he":
            self.W1 = r.normal(0, np.sqrt(2 / n_in), (n_in, n_hidden)).astype(np.float32)
            self.W2 = r.normal(0, np.sqrt(2 / n_hidden), (n_hidden, n_out)).astype(np.float32)
        elif init == "zeros":
            self.W1 = np.zeros((n_in, n_hidden), np.float32)
            self.W2 = np.zeros((n_hidden, n_out), np.float32)
        self.b1 = np.zeros(n_hidden, np.float32)
        self.b2 = np.zeros(n_out, np.float32)

    def params(self):
        return {"W1": self.W1, "b1": self.b1, "W2": self.W2, "b2": self.b2}

    def forward(self, X, relu=True):
        self.X = X
        self.Z1 = X @ self.W1 + self.b1
        self.relu = relu
        self.A1 = np.maximum(0, self.Z1) if relu else self.Z1
        self.Z2 = self.A1 @ self.W2 + self.b2
        self.P = softmax(self.Z2)
        return self.P

    def backward(self, y):
        B = len(y)
        dZ2 = self.P.copy()
        dZ2[np.arange(B), y] -= 1                       # P − Y without building the one-hot matrix
        dZ2 /= B
        grads = {"W2": self.A1.T @ dZ2, "b2": dZ2.sum(0)}
        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * (self.Z1 > 0) if self.relu else dA1
        grads["W1"] = self.X.T @ dZ1
        grads["b1"] = dZ1.sum(0)
        return grads

    def step(self, grads, lr):
        for k, p in self.params().items():
            p -= lr * grads[k]                           # in-place update

    def predict(self, X, relu=True):
        return self.forward(X, relu).argmax(1)

### Gradient check: is the backward pass right?

Backprop bugs are silent: the network still trains, just worse. Before trusting it, compare each analytic gradient against a **numerical** one:

$$\frac{\partial L}{\partial w} \approx \frac{L(w + \epsilon) - L(w - \epsilon)}{2\epsilon}$$

Too slow for training (two forward passes per weight!), perfect for checking a few weights.

In [ ]:
def gradient_check(n_checks=8, eps=1e-3):
    net = TwoLayerNet(n_in=20, n_hidden=7, n_out=4, seed=1)
    net.W1, net.W2 = net.W1.astype(np.float64), net.W2.astype(np.float64)   # float64 for an accurate check
    net.b1, net.b2 = net.b1.astype(np.float64) + 0.1, net.b2.astype(np.float64)
    X = rng.normal(size=(5, 20)); y = rng.integers(0, 4, 5)
    net.forward(X); grads = net.backward(y)
    for name, p in net.params().items():
        rel_errors = []
        for _ in range(n_checks):
            idx = tuple(rng.integers(0, s) for s in p.shape)
            old = p[idx]
            p[idx] = old + eps; lp = cross_entropy(net.forward(X), y)
            p[idx] = old - eps; lm = cross_entropy(net.forward(X), y)
            p[idx] = old
            num = (lp - lm) / (2 * eps)
            rel_errors.append(abs(num - grads[name][idx]) / max(abs(num) + abs(grads[name][idx]), 1e-12))
        print(f"{name}: max relative error {max(rel_errors):.2e}")

gradient_check()

Relative errors around $10^{-7}$ or smaller mean the backward pass is correct. Anything above ~$10^{-3}$ means a bug.

### Training loop: mini-batch SGD

Instead of computing the gradient on all 50,000 images (slow) or one image (noisy), use **mini-batches** of 64: a decent estimate of the downhill direction, computed fast. One pass through all the training data = one **epoch**.

In [ ]:
def accuracy(net, X, y, relu=True):
    return float((net.predict(X, relu) == y).mean())

def train(net, X, y, X_val, y_val, epochs=10, lr=0.1, batch_size=64, relu=True, verbose=True):
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        t0 = time.perf_counter()
        order = rng.permutation(len(X))                   # reshuffle every epoch
        for s in range(0, len(X), batch_size):
            idx = order[s:s + batch_size]
            net.forward(X[idx], relu)
            net.step(net.backward(y[idx]), lr)
        hist["train_loss"].append(cross_entropy(net.forward(X, relu), y))
        hist["train_acc"].append(accuracy(net, X, y, relu))
        hist["val_loss"].append(cross_entropy(net.forward(X_val, relu), y_val))
        hist["val_acc"].append(accuracy(net, X_val, y_val, relu))
        if verbose:
            print(f"epoch {epoch:2d} | train loss {hist['train_loss'][-1]:.4f} acc {hist['train_acc'][-1]:.4f} | "
                  f"val loss {hist['val_loss'][-1]:.4f} acc {hist['val_acc'][-1]:.4f} | {time.perf_counter() - t0:.1f}s")
    return hist

net = TwoLayerNet(n_hidden=128)
print(f"parameters: {sum(p.size for p in net.params().values()):,}")
history = train(net, X_train, y_train, X_val, y_val, epochs=10, lr=0.1)
plot_history(history)

About **97% validation accuracy** with ~100k parameters, written from scratch in a few dozen lines.

### What did the hidden neurons learn?
Each column of $W_1$ is 784 weights, one per pixel, so it can be shown as a 28×28 image. Red = "this pixel pushes me up", blue = "pushes me down".

In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(12, 4))
for ax, k in zip(axes.flat, range(30)):
    w = net.W1[:, k].reshape(28, 28)
    lim = np.abs(w).max()
    ax.imshow(w, cmap="RdBu_r", vmin=-lim, vmax=lim); ax.axis("off")
plt.suptitle("First 30 hidden neurons' weights"); plt.tight_layout(); plt.show()

You can make out blob- and stroke-shaped regions in the middle, but there's a lot of speckle. Why? **Pixels near the border are 0 in almost every MNIST image**, so their gradient ($X^\top \partial Z_1$) is 0 and those weights never move from their random starting values. Only pixels that actually vary get learned. Convolutions (module 03) learn far cleaner patterns with far fewer weights.

---
# 🚀 5 · Real tool check: PyTorch autograd agrees

PyTorch computes gradients automatically (**autograd**). Copy our weights into PyTorch, run the same forward pass, and compare its gradients with our hand-derived ones. Notebook 02b moves fully to PyTorch.

In [ ]:
import torch
import torch.nn.functional as F

xb, yb = X_train[:32], y_train[:32]
net.forward(xb); ours = net.backward(yb)

tW1, tb1, tW2, tb2 = (torch.tensor(p, requires_grad=True) for p in (net.W1, net.b1, net.W2, net.b2))
logits = torch.relu(torch.tensor(xb) @ tW1 + tb1) @ tW2 + tb2
loss = F.cross_entropy(logits, torch.tensor(yb))                # softmax + cross-entropy in one
loss.backward()                                                  # autograd fills .grad for every tensor

for name, t in [("W1", tW1), ("b1", tb1), ("W2", tW2), ("b2", tb2)]:
    print(f"{name}: max |ours − autograd| = {np.abs(ours[name] - t.grad.numpy()).max():.2e}")

Identical up to float32 rounding. Everything `loss.backward()` does, you just did by hand.

---
# 📊 6 · Evaluate

### 6.1 · Final test accuracy: once, at the end

In [ ]:
test_acc = accuracy(net, X_test, y_test)
print(f"validation accuracy: {history['val_acc'][-1]:.4f}")
print(f"TEST accuracy      : {test_acc:.4f}   ← the number you'd report")

Validation and test agree closely, a good sign we didn't overfit our choices to the validation set.

### 6.2 · Confusion matrix: *which* mistakes?
Row = true digit, column = predicted digit. The diagonal is correct; everything else is a specific confusion.

In [ ]:
pred_test = net.predict(X_test)
cm = np.zeros((10, 10), int)
np.add.at(cm, (y_test, pred_test), 1)                           # count every (true, predicted) pair

fig, ax = plt.subplots(figsize=(6.5, 5.5))
off = cm.copy(); np.fill_diagonal(off, 0)
im = ax.imshow(off, cmap="Reds")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="black" if i != j else "gray")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_title("Confusion matrix (color = errors only)"); plt.colorbar(im, ax=ax); plt.show()

per_class = cm.diagonal() / cm.sum(1)
worst = np.argsort(per_class)[:3]
print("per-class accuracy:", " ".join(f"{d}:{a:.3f}" for d, a in enumerate(per_class)))
print("hardest digits    :", worst.tolist())
i, j = np.unravel_index(off.argmax(), off.shape)
print(f"most common mistake: true {i} predicted as {j} ({off[i, j]} times)")

### 6.3 · Error gallery: look at the failures

The most useful evaluation habit: **look at the mistakes**, especially the ones the model was *most confident* about. They reveal label errors, ambiguous inputs, and blind spots.

In [ ]:
P_test = net.forward(X_test)
wrong = np.nonzero(pred_test != y_test)[0]
conf = P_test[wrong, pred_test[wrong]]
most_confident_wrong = wrong[np.argsort(conf)[::-1][:16]]

fig, axes = plt.subplots(2, 8, figsize=(13, 3.8))
for ax, i in zip(axes.flat, most_confident_wrong):
    ax.imshow(X_test[i].reshape(28, 28), cmap="gray"); ax.axis("off")
    ax.set_title(f"true {y_test[i]} → {pred_test[i]}\n({P_test[i, pred_test[i]]:.0%} sure)", fontsize=9)
plt.suptitle(f"Most confident mistakes ({len(wrong)} errors out of {len(y_test)})"); plt.tight_layout(); plt.show()

Several of these are genuinely ambiguous, or arguably mislabeled. A human would hesitate too. Others are real model weaknesses (unusual slant or thickness), which **data augmentation** (module 05) and **CNNs** (module 03) fix.

---
# 🧪 7 · Experiments: predict first, then run

Each experiment trains for only 3 epochs to stay fast.

**Experiment 1: learning rate.** Predict what happens with 0.001, 0.1 and 5.0.

In [ ]:
# Prediction: ...
for lr in [0.001, 0.1, 5.0]:
    with np.errstate(all="ignore"):
        h = train(TwoLayerNet(), X_train, y_train, X_val, y_val, epochs=3, lr=lr, verbose=False)
    print(f"lr={lr:<6} val acc per epoch: {[round(a, 3) for a in h['val_acc']]} | final train loss {h['train_loss'][-1]:.3f}")
# Too small: barely moves in 3 epochs. Too big: overshoots the valley and bounces around or blows up (nan).

**Experiment 2: remove ReLU.** Without the activation, $A_1 = Z_1$, so the network computes $X W_1 W_2 + \dots$, which is just one linear layer. Predict the accuracy drop.

In [ ]:
# Prediction: ...
h_lin = train(TwoLayerNet(), X_train, y_train, X_val, y_val, epochs=3, lr=0.1, relu=False, verbose=False)
h_relu = train(TwoLayerNet(), X_train, y_train, X_val, y_val, epochs=3, lr=0.1, relu=True, verbose=False)
print(f"no ReLU (= linear model): {h_lin['val_acc'][-1]:.4f}")
print(f"with ReLU               : {h_relu['val_acc'][-1]:.4f}")

**Experiment 3: initialize all weights to zero.** Predict: will it learn?

In [ ]:
# Prediction: ...
h_zero = train(TwoLayerNet(init="zeros"), X_train, y_train, X_val, y_val, epochs=3, lr=0.1, verbose=False)
print("zero init val acc per epoch:", [round(a, 3) for a in h_zero["val_acc"]])
# With zero weights every hidden neuron computes the same thing and receives the same gradient,
# so they stay identical forever (and with ReLU, the gradient into W1 is exactly zero).
# Random init "breaks the symmetry" so neurons can specialize.

**Experiment 4: hidden layer size.** 8, 32, 128 or 512 neurons: where do the gains flatten out?

In [ ]:
# Prediction: ...
for hsize in [8, 32, 128, 512]:
    t0 = time.perf_counter()
    h = train(TwoLayerNet(n_hidden=hsize), X_train, y_train, X_val, y_val, epochs=3, lr=0.1, verbose=False)
    print(f"hidden={hsize:>3}: val acc {h['val_acc'][-1]:.4f} | {time.perf_counter() - t0:.1f}s")

**Experiment 5: overfitting.** Train on only **500** images for many epochs. Predict how the train and validation curves will diverge.

In [ ]:
# Prediction: ...
small = rng.choice(len(X_train), 500, replace=False)
h_small = train(TwoLayerNet(n_hidden=256), X_train[small], y_train[small], X_val, y_val, epochs=60, lr=0.1, batch_size=32, verbose=False)
plot_history(h_small)
print(f"final train acc {h_small['train_acc'][-1]:.3f} vs val acc {h_small['val_acc'][-1]:.3f}")

Training accuracy hits 100% (it **memorizes** the 500 images) while validation accuracy stalls around 87%. Validation loss bottoms out around epoch 20 and then slowly **creeps up**. That gap is overfitting: more training only makes the model more sure of its memorized answers. The fixes come in 02b and module 05: more data, augmentation, regularization, and **early stopping** (stop at the validation-loss minimum).

---
# 📝 8 · Recap

**Cheat sheet**
- Neuron = weighted sum + bias → activation. Layers of neurons; **ReLU** between them makes depth useful.
- **Softmax** turns scores into probabilities; **cross-entropy** = −log(probability of the right answer).
- **Backprop** = chain rule backwards. Softmax+CE gradient is simply $(P - Y)/B$. ReLU passes gradient only where active.
- **SGD**: $W \leftarrow W - \eta \nabla L$ on mini-batches. The learning rate is the most important knob.
- **Gradient-check** any hand-written backward pass. Use **He init**; zero init can't break symmetry.
- **Train / val / test**: tune on validation, report test once.
- Evaluate beyond accuracy: **confusion matrix**, **per-class accuracy**, **most-confident mistakes**.
- **Overfitting** = train loss ↓, val loss ↑. More data, regularization, early stopping.

**Interview-style questions**
1. Why do we need a non-linear activation? What does a deep network become without one?
2. Derive the gradient of softmax + cross-entropy with respect to the logits.
3. Why subtract the max inside softmax?
4. What goes wrong if all weights start at zero?
5. Explain the roles of the training, validation and test sets. What's "test set leakage"?
6. How would you notice a bug in a hand-written backward pass?
7. Your model has 99% train and 85% validation accuracy. What's happening, and what would you try?

**Next:** `02b_pytorch_basics.ipynb`, the same network in PyTorch, plus the reusable training loop for the rest of the course.